# Process all radar days: L0 → L1 → L2

Run from top to bottom. Reuse the nearest-gate, 1 km remapping from `choose_dates_and_process_them` and `calc_rate_from_refl(..., offset=13)` from the inspection notebook.

- **L1:** measured reflectivity (`dBZ`), on a latitude/longitude grid.
- **L2:** rainfall rate (`rainfall_rate`, mm/hr), with the offset applied once.
- Keep native scan times and missing values; no time resampling or rainfall accumulation.
- Skip existing daily outputs, including old `l1_radar_YYYY_MM_DD.nc` names. Existing files are neither corrected nor overwritten.

A failed scan fails its L1 day; the next day still runs. L2 has its own loop and also processes older L1 files. Run only one copy of this notebook at a time.


## 1. Imports and settings


In [ ]:
from pathlib import Path
from contextlib import ExitStack
import re

import numpy as np
import pandas as pd
import xarray as xr
import dask.array as da
from scipy.spatial import cKDTree, ConvexHull, Delaunay
from pyproj import CRS, Geod, Transformer
from pacha.L2_processing import radar as l2rad


In [ ]:
l0_dir = Path('/media/admingosset/T7Shield/data/raw_data/ceara/ftp_evandro_2026/')
processed_dir = Path('../data/processed')
l1_dir = processed_dir / 'l1_radar'
l2_dir = processed_dir / 'l2_radar'

resolution_m = 1000
offset_db = 13
for folder in [l1_dir, l2_dir]:
    folder.mkdir(parents=True, exist_ok=True)

report = []


## 2. Find all L0 files

Use the same filename date slice `[4:18]` as the source notebook. Search subfolders too, without a year, rainfall, or scan-count filter. Files whose names cannot be parsed are reported and skipped.


In [ ]:
rows = []
for path in sorted(l0_dir.rglob('*')):
    if not path.is_file():
        continue
    try:
        time = pd.to_datetime(path.name[4:18], format='%Y%m%d%H%M%S')
        rows.append({'time': time, 'paths': path})
    except Exception as error:
        print(f'SKIP filename: {path.name} | {error}')

if not rows:
    raise ValueError(f'No dated L0 files found in {l0_dir}')
l0_files = pd.DataFrame(rows).set_index('time').sort_index()
l0_days = list(l0_files.index.normalize().unique())
print(f'{len(l0_files)} L0 files, {len(l0_days)} days')


## 3. Small helpers: skip, report, and save


In [ ]:
def existing_daily(folder, level, day):
    prefix = f'{level}_radar_{day:%Y_%m_%d}'
    files = sorted(folder.glob(f'{prefix}_*.nc'))
    legacy = folder / f'{prefix}.nc'
    if legacy.exists():
        files.append(legacy)
    if len(files) > 1:
        raise ValueError(f'Multiple {level} files for {day.date()}: {files}')
    return files[0] if files else None


def record(stage, day, status, detail):
    report.append(dict(stage=stage, day=str(day.date()),
                       status=status, detail=str(detail)))
    print(f'{stage} | {day.date()} | {status} | {detail}')


Filenames use the source day, first/last **scan midpoint**, median spacing in minutes, and actual number of time steps. Example: `l2_radar_2024_04_16_000230-235730_step5min_n288.nc`.

Irregular times add `_irregular`; one time step uses `stepNAmin`. The full start/stop timestamps and spacing definition are also saved as attributes. Gaps are not filled.


In [ ]:
def time_details(ds):
    times = pd.DatetimeIndex(pd.to_datetime(ds.time.values))
    if len(times) == 0 or times.hasnans or not times.is_monotonic_increasing:
        raise ValueError('Expected nonempty, valid, sorted times')
    if times.has_duplicates:
        raise ValueError('Duplicate scan midpoints')
    steps = np.diff(times.asi8) / 60e9
    step = float(np.median(steps)) if len(steps) else np.nan
    irregular = bool(len(steps) and not np.allclose(steps, step))
    token = f'{step:.6g}' if np.isfinite(step) else 'NA'
    suffix = f'{times[0]:%H%M%S}-{times[-1]:%H%M%S}_step{token}min_n{len(times)}'
    suffix += '_irregular' if irregular else ''
    attrs = dict(time_coverage_start=times[0].isoformat(),
                 time_coverage_end=times[-1].isoformat(), time_zone='UTC',
                 step_minutes=step, nsteps=len(times), irregular_sampling=int(irregular),
                 time_range=f'{times[0].isoformat()}/{times[-1].isoformat()}',
                 step_definition='median spacing of scan midpoints; no resampling')
    return suffix, attrs


In [ ]:
def save_daily(ds, folder, level, day):
    suffix, attrs = time_details(ds)
    ds.attrs.update(attrs, source_day=str(day.date()))
    output = folder / f'{level}_radar_{day:%Y_%m_%d}_{suffix}.nc'
    temporary = output.with_suffix('.nc.part')
    if existing_daily(folder, level, day):
        raise FileExistsError(f'A daily {level} file already exists')
    try:
        ds.to_netcdf(temporary)
        temporary.rename(output)
    finally:
        temporary.unlink(missing_ok=True)
    return output


## 4. Open one native day

Use the midpoint of each L0 file’s `t_i` / `t_f`. Keep files open while the lazy daily result is written, then close them with `ExitStack`. Require the same native grid for every scan in a day.


In [ ]:
def open_daily_funceme(paths, stack):
    scans = []
    for path in paths:
        ds = stack.enter_context(xr.open_dataset(path, chunks={}))
        start, stop = pd.to_datetime([ds.attrs['t_i'], ds.attrs['t_f']], utc=True)
        if pd.isna(start) or pd.isna(stop) or stop < start:
            raise ValueError(f'Invalid scan interval: {path}')
        midpoint = (start + (stop - start) / 2).tz_localize(None)
        scan = ds[['dBZ', 'xlon', 'ylat']]
        if 'time' in scan.dims:
            scan = scan.squeeze('time', drop=True)
        scan = scan.assign(dBZ=scan.dBZ.expand_dims(time=[midpoint]))
        scans.append(scan)
    daily = xr.concat(scans, dim='time', data_vars='minimal',
                      coords='minimal', compat='equals', join='exact').sortby('time')
    daily['dBZ'] = daily.dBZ.astype('float32').where(daily.dBZ != -100)
    time_details(daily)
    return daily


## 5. The same spatial remapping, split into short cells

Keep nearest-gate sampling, the convex-hull footprint mask, and source NaNs. The helper below replaces the long `to_lonlat` cell in the original notebook.


In [ ]:
def native_coordinates(ds):
    lon, lat = np.asarray(ds.xlon.values), np.asarray(ds.ylat.values)
    dims = ds.xlon.dims
    if lon.ndim != 2 or lon.shape != lat.shape or ds.ylat.dims != dims:
        raise ValueError('Expected matching static 2-D xlon/ylat')
    if not (np.isfinite(lon).all() and np.isfinite(lat).all()):
        raise ValueError('Native coordinates contain missing values')
    if set(ds.dBZ.dims) != {'time', *dims}:
        raise ValueError('Expected dBZ(time, native_dim_1, native_dim_2)')
    return lon, lat, dims


In [ ]:
def target_grid(lon, lat, resolution_m):
    if not np.isfinite(resolution_m) or resolution_m <= 0:
        raise ValueError('resolution_m must be positive and finite')
    lon0, lat0 = float(np.median(lon)), float(np.median(lat))
    geod = Geod(ellps='WGS84')
    _, _, dx = geod.inv(lon0, lat0, lon0 + 0.01, lat0)
    _, _, dy = geod.inv(lon0, lat0, lon0, lat0 + 0.01)
    dlon, dlat = resolution_m / (dx / 0.01), resolution_m / (dy / 0.01)
    out_lon = lon.min() + np.arange(int(np.ptp(lon) / dlon) + 1) * dlon
    out_lat = lat.min() + np.arange(int(np.ptp(lat) / dlat) + 1) * dlat
    if len(out_lon) * len(out_lat) > 25_000_000:
        raise ValueError('Target grid too large; increase resolution_m')
    return out_lon, out_lat, lon0, lat0


In [ ]:
def nearest_gates(lon, lat, out_lon, out_lat, lon0, lat0):
    local_crs = CRS.from_proj4(
        f'+proj=aeqd +lat_0={lat0} +lon_0={lon0} +datum=WGS84 +units=m'
    )
    project = Transformer.from_crs('EPSG:4326', local_crs, always_xy=True)
    sx, sy = project.transform(lon.ravel(), lat.ravel())
    source = np.column_stack([sx, sy])
    target_lon, target_lat = np.meshgrid(out_lon, out_lat)
    tx, ty = project.transform(target_lon.ravel(), target_lat.ravel())
    target = np.column_stack([tx, ty])
    _, nearest = cKDTree(source).query(target)
    hull = ConvexHull(source)
    inside = Delaunay(source[hull.vertices]).find_simplex(target) >= 0
    return nearest, inside


In [ ]:
def remap_field(field, dims, nearest, inside, ny, nx):
    field = field.transpose('time', *dims)
    native_chunks = field.encoding.get('chunksizes')
    read_chunk = native_chunks[0] if native_chunks else 1
    data = field.chunk({'time': read_chunk, dims[0]: -1, dims[1]: -1})
    data = data.data.rechunk({0: 1})

    def remap(block):
        values = block.reshape(block.shape[0], -1)[:, nearest].copy()
        values[:, ~inside] = np.nan
        return values.reshape(block.shape[0], ny, nx)

    return da.map_blocks(
        remap, data, dtype=field.dtype, chunks=(data.chunks[0], (ny,), (nx,)),
        drop_axis=(1, 2), new_axis=(1, 2),
        meta=np.empty((0, 0, 0), dtype=field.dtype)
    )


In [ ]:
def to_lonlat(ds, resolution_m=1000):
    lon, lat, dims = native_coordinates(ds)
    out_lon, out_lat, lon0, lat0 = target_grid(lon, lat, resolution_m)
    nearest, inside = nearest_gates(lon, lat, out_lon, out_lat, lon0, lat0)
    mapped = remap_field(ds.dBZ, dims, nearest, inside, len(out_lat), len(out_lon))
    result = xr.Dataset(
        {'dBZ': (('time', 'lat', 'lon'), mapped, dict(ds.dBZ.attrs))},
        coords={'time': ds.time, 'lat': out_lat, 'lon': out_lon}, attrs=dict(ds.attrs)
    )
    result.lon.attrs = dict(standard_name='longitude', units='degrees_east', axis='X')
    result.lat.attrs = dict(standard_name='latitude', units='degrees_north', axis='Y')
    result.dBZ.attrs.update(units='dBZ')
    result.attrs.update(nominal_grid_spacing_m=float(resolution_m),
                        regridding='nearest native gate; source NaNs retained')
    for name in ['t_i', 't_f']:
        if name in result.attrs:
            result.attrs[f'source_first_scan_{name}'] = result.attrs.pop(name)
    return result


## 6. L0 → L1: process every missing day

An unreadable scan, grid mismatch, or write error is reported for that day. Temporary files are removed; no incomplete daily output is kept.


In [ ]:
for day in l0_days:
    try:
        existing = existing_daily(l1_dir, 'l1', day)
        if existing:
            record('L1', day, 'skip', existing.name)
            continue
        paths = l0_files.loc[l0_files.index.normalize() == day, 'paths']
        with ExitStack() as stack:
            native = open_daily_funceme(paths, stack)
            l1 = to_lonlat(native, resolution_m=resolution_m)
            l1.attrs.update(processing_level='L1', source_file_count=len(paths))
            output = save_daily(l1, l1_dir, 'l1', day)
        record('L1', day, 'saved', output.name)
    except Exception as error:
        record('L1', day, 'error', f'{type(error).__name__}: {error}')


## 7. Find all L1 days, including previously processed files


In [ ]:
l1_days = set()
for path in sorted(l1_dir.glob('l1_radar_*.nc')):
    try:
        match = re.match(r'^l1_radar_(\d{4}_\d{2}_\d{2})(?:_|\.nc$)', path.name)
        if match is None:
            raise ValueError('Unrecognized daily L1 filename')
        l1_days.add(pd.to_datetime(match.group(1), format='%Y_%m_%d'))
    except Exception as error:
        print(f'SKIP L1 filename: {path.name} | {error}')

print(f'{len(l1_days)} L1 days available for L2')


## 8. L1 → L2: rainfall with offset = 13 dB

Pass the keyword directly to the existing Pacha function; do not manually add the offset to L1. The output is a **rate in mm/hr**, so no ×6 factor is needed here (that factor was only for the 10-minute gauge accumulations).


In [ ]:
def rainfall_from_l1(ds):
    reflectivity = ds.dBZ.where(ds.dBZ != -100)
    rate = l2rad.calc_rate_from_refl(reflectivity, offset=offset_db)
    rate = rate.where(reflectivity.notnull()).rename('rainfall_rate')
    rate.attrs = dict(long_name='Radar rainfall rate', units='mm/hr')
    result = rate.to_dataset()
    result.attrs = dict(ds.attrs)
    result.attrs.update(processing_level='L2', reflectivity_offset_db=float(offset_db),
                        rainfall_conversion='pacha.L2_processing.radar.calc_rate_from_refl',
                        offset_application='offset keyword; applied once at L1 to L2')
    return result


In [ ]:
for day in sorted(l1_days):
    try:
        existing = existing_daily(l2_dir, 'l2', day)
        if existing:
            record('L2', day, 'skip', existing.name)
            continue
        source = existing_daily(l1_dir, 'l1', day)
        with xr.open_dataset(source, chunks={'time': 1}) as ds:
            l2 = rainfall_from_l1(ds.sortby('time'))
            l2.attrs['source_l1_file'] = source.name
            output = save_daily(l2, l2_dir, 'l2', day)
        record('L2', day, 'saved', output.name)
    except Exception as error:
        record('L2', day, 'error', f'{type(error).__name__}: {error}')


## 9. Check the run

Rerun to retry failed days and produce newly missing files. A daily file that already exists is always skipped, even if L0 inputs or settings have since changed; inspect/remove that output yourself if you intend to rebuild it.


In [ ]:
run_report = pd.DataFrame(report, columns=['stage', 'day', 'status', 'detail'])
print(run_report.groupby(['stage', 'status']).size().to_string())
run_report.loc[run_report['status'] == 'error']
